# Lightricks LTX-2.5 (22B) Dual-Stream DiT Video & Audio Generation Pipeline

此 Notebook 在 **Google Colab (A100 / H100)** 上執行真正的 **Lightricks/LTX-2.5 (22B)** 旗艦級模型。
- **Transformer**: `diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors` (39.1 GB)
- **Text Encoder**: `text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors` (24.5 GB)
- **Video VAE**: `vae/ltx-2.5-video-vae-bf16.safetensors` (1.37 GB)
- **Audio VAE**: `vae/ltx-2.5-audio-vae-bf16.safetensors` (340 MB) 原生同步音訊生成
- **Duration Head**: `model_patches/ltx-2.5-duration-head-bf16.safetensors` (3.7 MB)
- **Latent Spatial Upscaler**: `latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors` (930 MB) 兩階段 2x 超解析


## 1. 檢查 GPU 硬體與儲存空間

In [ ]:
!nvidia-smi
!df -h /content


## 2. 環境建置 (安裝 uv 與 Clone 官方 Lightricks/LTX-2)

In [ ]:
import os, subprocess, sys

print('[1/3] 正在安裝 uv 與 huggingface_hub[cli]...')
subprocess.run(['pip', 'install', '-q', 'uv', 'huggingface_hub[cli]'], check=True)

repo_dir = '/content/LTX-2'
if not os.path.exists(repo_dir):
    print('[2/3] 正在 Clone 官方 Lightricks/LTX-2 倉庫...')
    subprocess.run(['git', 'clone', 'https://github.com/Lightricks/LTX-2.git', repo_dir], check=True)
else:
    print('[2/3] Lightricks/LTX-2 倉庫已就緒。')

print('[3/3] 透過 uv 同步建立完整依賴環境 (含 natten 加速支援)...')
subprocess.run(['uv', 'sync', '--extra', 'natten'], cwd=repo_dir, check=True)
subprocess.run(['uv', 'pip', 'install', 'matplotlib-inline'], cwd=repo_dir)
print('✅ LTX-2.5 環境建置完成！')


## 3. 下載 LTX-2.5 22B 官方權重 (已驗證 Hugging Face 授權)

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

hf_token = os.environ.get('HF_TOKEN', '').strip()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass

assert hf_token, '需設定 HF_TOKEN 才能下載已獲授權的 Lightricks/LTX-2.5 模型！'

model_repo = 'Lightricks/LTX-2.5'
base_dir = Path('/content/models/ltx-2.5')

files = [
    'diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors',
    'text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors',
    'vae/ltx-2.5-video-vae-bf16.safetensors',
    'vae/ltx-2.5-audio-vae-bf16.safetensors',
    'model_patches/ltx-2.5-duration-head-bf16.safetensors',
    'latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors',
]

print(f'正在檢查/下載 {len(files)} 個 LTX-2.5 22B 核心權重檔案 (共約 66 GB)...')
for rel_path in files:
    target_file = base_dir / rel_path
    if target_file.exists() and target_file.stat().st_size > 1024 * 1024:
        sz_gb = round(target_file.stat().st_size / (1024**3), 2)
        print(f'  [已存在] {rel_path} ({sz_gb} GB)')
    else:
        print(f'  [正在高速下載] {rel_path} ...')
        target_file.parent.mkdir(parents=True, exist_ok=True)
        hf_hub_download(
            repo_id=model_repo,
            filename=rel_path,
            local_dir=str(base_dir),
            token=hf_token
        )
        sz_gb = round(target_file.stat().st_size / (1024**3), 2)
        print(f'  ✅ 下載完成: {rel_path} ({sz_gb} GB)')

print('🎉 所有 LTX-2.5 22B 權重均已就緒！')


## 4. 讀取任務參數與提示詞

In [ ]:
import os
from pathlib import Path

prompt_file = os.environ.get('LTX_PROMPT_FILE', '/content/prompt.txt')
image_file = os.environ.get('LTX_IMAGE_FILE', '').strip()
output_path = os.environ.get('LTX_OUTPUT_PATH', '/content/output.mp4')
num_frames = int(os.environ.get('LTX_NUM_FRAMES', '121'))
seed = int(os.environ.get('LTX_SEED', '42'))

# LTX-2.5 要求幀數必須滿足 (frames % 8 == 1)
if num_frames % 8 != 1:
    old_frames = num_frames
    num_frames = ((num_frames - 1) // 8) * 8 + 1
    if num_frames < 9:
        num_frames = 121
    print(f'調整影格數: {old_frames} -> {num_frames} 以符合 (frames % 8 == 1) 約束')

if os.path.exists(prompt_file):
    with open(prompt_file, 'r', encoding='utf-8') as f:
        prompt = f.read().strip()
else:
    prompt = os.environ.get('LTX_PROMPT', 'A colossal oriental golden giant dragon roaring fiercely in turbulent storm clouds, lightning, cinematic 4K')

print('=' * 60)
print(f'提示詞: {prompt}')
print(f'參考圖片: {image_file if image_file and os.path.exists(image_file) else "(純文字 Text-to-Video)"}')
print(f'總影格數: {num_frames} 幀 (約 {round(num_frames/24, 2)} 秒 @ 24fps)')
print(f'隨機種子 Seed: {seed}')
print(f'輸出目標: {output_path}')
print('=' * 60)


## 5. 執行 LTX-2.5 22B Distilled Pipeline 推論

In [ ]:
import os, subprocess, sys
from pathlib import Path

Path(output_path).parent.mkdir(parents=True, exist_ok=True)

cmd = [
    'uv', 'run', 'python', '-m', 'ltx_pipelines.distilled',
    '--transformer-path', '/content/models/ltx-2.5/diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors',
    '--text-encoder-path', '/content/models/ltx-2.5/text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors',
    '--video-vae-path', '/content/models/ltx-2.5/vae/ltx-2.5-video-vae-bf16.safetensors',
    '--audio-vae-path', '/content/models/ltx-2.5/vae/ltx-2.5-audio-vae-bf16.safetensors',
    '--duration-head-path', '/content/models/ltx-2.5/model_patches/ltx-2.5-duration-head-bf16.safetensors',
    '--spatial-upsampler-path', '/content/models/ltx-2.5/latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors',
    '--prompt', prompt,
    '--seed', str(seed),
    '--num-frames', str(num_frames),
    '--output-path', str(output_path),
]

if image_file and os.path.exists(image_file):
    print(f'加入首格圖像條件引導: {image_file}')
    cmd.extend(['--image', str(image_file), '0', '1.0'])

opt_h = os.environ.get('LTX_HEIGHT', '').strip()
opt_w = os.environ.get('LTX_WIDTH', '').strip()
if opt_h and opt_w:
    cmd.extend(['--height', str(opt_h), '--width', str(opt_w)])

# 修復 Colab 中 MPLBACKEND 導致 matplotlib 初始化崩潰的問題
sub_env = os.environ.copy()
sub_env['MPLBACKEND'] = 'Agg'

print('\n正在啟動 LTX-2.5 22B 影音雙串流推論...')
print('完整指令:', ' '.join(cmd))
sys.stdout.flush()

proc = subprocess.Popen(
    cmd,
    cwd='/content/LTX-2',
    env=sub_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'LTX-2.5 22B 推論異常失敗 (結束代碼 {proc.returncode})')

print('\n🎉 LTX-2.5 22B 推論成功完成！')


## 6. 驗證產出影片與音訊串流

In [ ]:
import os, subprocess, json

assert os.path.exists(output_path), f'未找到生成輸出檔案: {output_path}'
size_mb = round(os.path.getsize(output_path) / (1024 * 1024), 2)
print(f'[檔案大小] {size_mb} MB')

res = subprocess.run(
    ['ffprobe', '-v', 'error', '-show_entries', 'stream=codec_type', '-of', 'json', output_path],
    capture_output=True,
    text=True
)
data = json.loads(res.stdout)
streams = [s.get('codec_type') for s in data.get('streams', [])]
print(f'[偵測到串流] {streams}')
assert 'video' in streams, '輸出檔案缺少視訊串流！'
print('狀態：LTX-2.5 22B 影音驗證成功！')
